In [1]:
!pip install https://test.pypi.org/simple/ supervision==0.3.0
!pip install transformers
!pip install pytorch-lightning
!pip install roboflow
!pip install timm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 2.6 MB/s eta 0:00:0000:0100:01
  ERROR: Cannot unpack file /tmp/pip-unpack-9_p1971b/simple.html (downloaded from /tmp/pip-req-build-wbswwv7r, content-type: text/html); cannot detect archive format
ERROR: Cannot determine archive format of /tmp/pip-req-build-wbswwv7r
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 3.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.2/869.2 kB 6.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 2.3 MB/s eta 0:00:0000:0100:01


In [3]:
!pip install supervision
import torch
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)

import roboflow
import supervision
import transformers
import pytorch_lightning

print(
    "roboflow:", roboflow.__version__, 
    "; supervision:", supervision.__version__, 
    "; transformers:", transformers.__version__, 
    "; pytorch_lightning:", pytorch_lightning.__version__
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 1.2 MB/s eta 0:00:00ta 0:00:01
/bin/bash: line 1: nvcc: command not found
torch:  2.4 ; cuda:  2.4.1
roboflow: 1.1.47 ; supervision: 0.24.0 ; transformers: 4.45.2 ; pytorch_lightning: 2.4.0


In [4]:
%cd {HOME}
!wget https://media.roboflow.com/notebooks/examples/dog.jpeg

[Errno 2] No such file or directory: '{HOME}'
/home/ahabb
--2024-10-08 16:59:53--  https://media.roboflow.com/notebooks/examples/dog.jpeg
Resolving media.roboflow.com (media.roboflow.com)... 34.110.133.209
Connecting to media.roboflow.com (media.roboflow.com)|34.110.133.209|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 106055 (104K) [image/jpeg]
Saving to: ‘dog.jpeg’

dog.jpeg            100%[===================>] 103.57K   515KB/s    in 0.2s    

2024-10-08 16:59:54 (515 KB/s) - ‘dog.jpeg’ saved [106055/106055]



In [5]:
IMAGE_PATH = '/home/ahabb/Downloads/fod-a/FullDatasetV.2.1-400x400/Battery1/frame/frame_000002.PNG'

In [6]:
import torch
from transformers import DetrForObjectDetection, DetrImageProcessor


# settings
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = 'facebook/detr-resnet-50'
CONFIDENCE_TRESHOLD = 0.5
IOU_TRESHOLD = 0.8

image_processor = DetrImageProcessor.from_pretrained(CHECKPOINT)
model = DetrForObjectDetection.from_pretrained(CHECKPOINT)
model.to(DEVICE)

Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


DetrForObjectDetection(
  (model): DetrModel(
    (backbone): DetrConvModel(
      (conv_encoder): DetrConvEncoder(
        (model): FeatureListNet(
          (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
          (bn1): DetrFrozenBatchNorm2d()
          (act1): ReLU(inplace=True)
          (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
          (layer1): Sequential(
            (0): Bottleneck(
              (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (bn1): DetrFrozenBatchNorm2d()
              (act1): ReLU(inplace=True)
              (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
              (bn2): DetrFrozenBatchNorm2d()
              (drop_block): Identity()
              (act2): ReLU(inplace=True)
              (aa): Identity()
              (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      

In [13]:
import cv2
import torch
import xml.etree.ElementTree as ET
import supervision as sv
import matplotlib.pyplot as plt
from transformers import DetrForObjectDetection, DetrImageProcessor

# Settings
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = 'facebook/detr-resnet-50'
CONFIDENCE_THRESHOLD = 0.5  # Corrected spelling here
IOU_THRESHOLD = 0.8

# Load the image processor and model
image_processor = DetrImageProcessor.from_pretrained(CHECKPOINT)
model = DetrForObjectDetection.from_pretrained(CHECKPOINT)
model.to(DEVICE)

# Load image
IMAGE_PATH = '/home/ahabb/Downloads/fod-a/FullDatasetV.2.1-400x400/Battery1/frame/frame_000002.PNG'
XML_PATH = '/home/ahabb/Downloads/fod-a/FullDatasetV.2.1-400x400/Battery1/Annotations/frame_000002.xml'

# Function to read XML annotations
def read_xml_annotations(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    annotations = []
    
    # Assuming the structure contains bounding boxes
    for object_elem in root.findall('object'):
        class_name = object_elem.find('name').text
        bndbox = object_elem.find('bndbox')
        
        # Convert bounding box coordinates to float and then to int
        x_min = int(float(bndbox.find('xmin').text))
        y_min = int(float(bndbox.find('ymin').text))
        x_max = int(float(bndbox.find('xmax').text))
        y_max = int(float(bndbox.find('ymax').text))

        annotations.append({
            'class_name': class_name,
            'bbox': [x_min, y_min, x_max, y_max]
        })
    
    return annotations

# Step 1: Load ground truth from XML
annotations = read_xml_annotations(XML_PATH)
print("Extracted Annotations:", annotations)

# Step 2: Load image and run prediction
image = cv2.imread(IMAGE_PATH)

with torch.no_grad():
    # Preprocess image and predict
    inputs = image_processor(images=image, return_tensors='pt').to(DEVICE)
    outputs = model(**inputs)

    # Post-process predictions
    target_sizes = torch.tensor([image.shape[:2]]).to(DEVICE)
    results = image_processor.post_process_object_detection(
        outputs=outputs, 
        threshold=CONFIDENCE_THRESHOLD,  
        target_sizes=target_sizes
    )[0]

# Annotate the detections
detections = sv.Detections.from_transformers(transformers_results=results)

# Check for attributes and create labels
if hasattr(detections, 'scores') and hasattr(detections, 'class_ids'):
    labels = [
        f"{model.config.id2label[class_id]} {score:0.2f}" 
        for score, class_id in zip(detections.scores, detections.class_ids)
    ]
else:
    print("The expected attributes 'scores' or 'class_ids' are not found in the detections object.")
    labels = []

# Annotate boxes on the image if labels are created
if labels:
    box_annotator = sv.BoxAnnotator()
    frame = box_annotator.annotate(scene=image, detections=detections, labels=labels)

    # Display the result
    plt.figure(figsize=(16, 16))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')  # Hide axes
    plt.title("Detected Objects")
    plt.show()
else:
    print("No detections to display.")

# Step 3: Compare predictions with ground truth
for annotation in annotations:
    print(f"Ground Truth Class: {annotation['class_name']}, BBox: {annotation['bbox']}")


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Extracted Annotations: [{'class_name': 'Battery', 'bbox': [184, 140, 201, 199]}]
The expected attributes 'scores' or 'class_ids' are not found in the detections object.
No detections to display.
Ground Truth Class: Battery, BBox: [184, 140, 201, 199]


In [14]:
import cv2
import torch
import xml.etree.ElementTree as ET
import supervision as sv
import matplotlib.pyplot as plt
from transformers import DetrForObjectDetection, DetrImageProcessor

# Settings
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = 'facebook/detr-resnet-50'
CONFIDENCE_THRESHOLD = 0.5  # Corrected spelling here
IOU_THRESHOLD = 0.8

# Load the image processor and model
image_processor = DetrImageProcessor.from_pretrained(CHECKPOINT)
model = DetrForObjectDetection.from_pretrained(CHECKPOINT)
model.to(DEVICE)

# Load image
IMAGE_PATH = '/home/ahabb/Downloads/fod-a/FullDatasetV.2.1-400x400/Battery1/frame/frame_000002.PNG'
XML_PATH = '/home/ahabb/Downloads/fod-a/FullDatasetV.2.1-400x400/Battery1/Annotations/frame_000002.xml'

# Function to read XML annotations
def read_xml_annotations(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    annotations = []
    
    for object_elem in root.findall('object'):
        class_name = object_elem.find('name').text
        bndbox = object_elem.find('bndbox')
        
        # Convert bounding box coordinates to float and then to int
        x_min = int(float(bndbox.find('xmin').text))
        y_min = int(float(bndbox.find('ymin').text))
        x_max = int(float(bndbox.find('xmax').text))
        y_max = int(float(bndbox.find('ymax').text))

        annotations.append({
            'class_name': class_name,
            'bbox': [x_min, y_min, x_max, y_max]
        })
    
    return annotations

# Step 1: Load ground truth from XML
annotations = read_xml_annotations(XML_PATH)
print("Extracted Annotations:", annotations)

# Step 2: Load image and run prediction
image = cv2.imread(IMAGE_PATH)

with torch.no_grad():
    # Preprocess image and predict
    inputs = image_processor(images=image, return_tensors='pt').to(DEVICE)
    outputs = model(**inputs)

    # Debugging: print outputs
    print("Model Outputs:", outputs)

    # Post-process predictions
    target_sizes = torch.tensor([image.shape[:2]]).to(DEVICE)
    results = image_processor.post_process_object_detection(
        outputs=outputs, 
        threshold=CONFIDENCE_THRESHOLD,  
        target_sizes=target_sizes
    )[0]

# Annotate the detections
try:
    detections = sv.Detections.from_transformers(transformers_results=results)

    # Debugging: print detections
    print("Detections:", detections)

    # Check for attributes and create labels
    if hasattr(detections, 'scores') and hasattr(detections, 'class_ids'):
        labels = [
            f"{model.config.id2label[class_id]} {score:0.2f}" 
            for score, class_id in zip(detections.scores, detections.class_ids)
        ]
    else:
        print("The expected attributes 'scores' or 'class_ids' are not found in the detections object.")
        labels = []

    # Annotate boxes on the image if labels are created
    if labels:
        box_annotator = sv.BoxAnnotator()
        frame = box_annotator.annotate(scene=image, detections=detections, labels=labels)

        # Display the result
        plt.figure(figsize=(16, 16))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.axis('off')  # Hide axes
        plt.title("Detected Objects")
        plt.show()
    else:
        print("No detections to display.")

except Exception as e:
    print(f"Error during detection processing: {e}")

# Step 3: Compare predictions with ground truth
for annotation in annotations:
    print(f"Ground Truth Class: {annotation['class_name']}, BBox: {annotation['bbox']}")


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Extracted Annotations: [{'class_name': 'Battery', 'bbox': [184, 140, 201, 199]}]
Model Outputs: DetrObjectDetectionOutput(loss=None, loss_dict=None, logits=tensor([[[-14.8443,   0.3111,  -3.9911,  ...,  -9.1168,  -4.6866,   8.6399],
         [-15.3941,  -0.6619,  -4.2857,  ..., -11.8234,  -3.0429,   9.8280],
         [-15.3882,  -0.3409,  -4.7572,  ...,  -7.4016,  -5.9648,   8.1091],
         ...,
         [-14.9882,  -0.8726,  -5.0345,  ...,  -9.2454,  -6.0784,   7.8025],
         [-15.6870,  -1.0059,  -4.1122,  ..., -10.7802,  -7.4733,   8.1178],
         [-16.2689,  -0.9045,  -4.2794,  ..., -11.0405,  -3.2482,   9.8788]]],
       device='cuda:0'), pred_boxes=tensor([[[0.0027, 0.5747, 0.0054, 0.8125],
         [0.4840, 0.4207, 0.0335, 0.1312],
         [0.1122, 0.9847, 0.2260, 0.0310],
         [0.4873, 0.4048, 0.0307, 0.1061],
         [0.4848, 0.3499, 0.0570, 0.2882],
         [0.4562, 0.4464, 0.0672, 0.2064],
         [0.4817, 0.4648, 0.0288, 0.0499],
         [0.4785, 0.4202, 0.0